<a href="https://colab.research.google.com/github/marlon-2006/Tarea_Negocios_Digilaes_SI/blob/main/NegociosDigitales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Taller: Auditoría de Desempeño para Negocios Digitales:**

**Parte 1: Análisis de datos sintéticos**

En esta parte vamos a analizar un dataset de transacciones y determinar si la estrategia de M-Commerce (App Móvil) es más rentable que el E-Commerce tradicional (Web Desktop).

Vamos a generar una base de datos aleatoria, que simula la salida de un TPS.


In [28]:
import pandas as pd
import numpy as np


data = {
    'ID_Transaccion': range(1001, 1051),
    'Canal': np.random.choice(['Web Desktop', 'App Móvil'], 50, p=[0.4, 0.6]),
    'Metodo_Pago': np.random.choice(['Tarjeta Crédito', 'PayPal', 'Transferencia'], 50),
    'Total_Venta': np.random.randint(20, 500, 50), # Ventas entre $20 y $500
    'Categoria': np.random.choice(['Electrónica', 'Moda', 'Hogar'], 50)
}

df = pd.DataFrame(data)

df

,ID_Transaccion,Canal,Metodo_Pago,Total_Venta,Categoria
0,1001,App Móvil,Transferencia,118,Hogar
1,1002,App Móvil,PayPal,308,Moda
2,1003,App Móvil,PayPal,300,Hogar
3,1004,App Móvil,Transferencia,444,Hogar
4,1005,Web Desktop,Tarjeta Crédito,375,Hogar
5,1006,App Móvil,Transferencia,151,Electrónica
6,1007,Web Desktop,Tarjeta Crédito,164,Moda
7,1008,Web Desktop,PayPal,127,Electrónica
8,1009,Web Desktop,PayPal,64,Moda
9,1010,Web Desktop,Tarjeta Crédito,313,Hogar


**¿Qué preguntas deberíamos plantearnos para obtener las respuestas que buscamos?**
Buscamos saber cuál de las dos estrategias (M-Commerce o E-Commerce) es la más rentable.

Vamos a crear 4 KPIs para un MIS que nos permitan responder esta pregunta.

In [29]:
# Parte 1: Creación de KPIs para evaluar la rentabilidad por Canal

# 1. Ventas Totales por Canal
ventas_totales = df.groupby('Canal')['Total_Venta'].sum()

# 2. Ticket Promedio (Venta Promedio por Transacción)
ticket_promedio = df.groupby('Canal')['Total_Venta'].mean()

# 3. Número de Transacciones (Volumen)
volumen_transacciones = df.groupby('Canal')['ID_Transaccion'].count()

# 4. Transacciones de Alto Valor (Por ejemplo, ventas mayores a $250)
alto_valor = df[df['Total_Venta'] > 250].groupby('Canal')['ID_Transaccion'].count()

mis_reporte = pd.DataFrame({
    'Ventas Totales ($)': ventas_totales,
    'Ticket Promedio ($)': ticket_promedio.round(2),
    'Total Transacciones': volumen_transacciones,
    'Transacciones > $250': alto_valor
})

print("=== REPORTE MIS: M-COMMERCE VS. E-COMMERCE ===")
print(mis_reporte)
print("\n=== CONCLUSIÓN PRELIMINAR ===")
canal_lider = ventas_totales.idxmax()
print(f"El canal más rentable actualmente en términos de volumen total de ventas es: {canal_lider}")

=== REPORTE MIS: M-COMMERCE VS. E-COMMERCE ===
             Ventas Totales ($)  Ticket Promedio ($)  Total Transacciones  \
Canal                                                                       
App Móvil                  7767               242.72                   32   
Web Desktop                5367               298.17                   18   

             Transacciones > $250  
Canal                              
App Móvil                      15  
Web Desktop                    12  

=== CONCLUSIÓN PRELIMINAR ===
El canal más rentable actualmente en términos de volumen total de ventas es: App Móvil


**Parte 2: KPIs para E-Commerce**

Para esta parte vamos a utilizar los datos qu se encuentran en el repositorio en /data/ecommerce.zip

In [30]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')


data = pd.read_csv('/content/drive/MyDrive/ecommerce.zip')
data

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,ID,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N
0,1,D,Flight,4,2,177,3,low,F,44,1233,1
1,2,F,Flight,4,5,216,2,low,M,59,3088,1
2,3,A,Flight,2,2,183,4,low,M,48,3374,1
3,4,B,Flight,3,3,176,4,medium,M,10,1177,1
4,5,C,Flight,2,2,184,3,medium,F,46,2484,1
...,...,...,...,...,...,...,...,...,...,...,...,...
10994,10995,A,Ship,4,1,252,5,medium,F,1,1538,1
10995,10996,B,Ship,4,1,232,5,medium,F,6,1247,0
10996,10997,C,Ship,5,4,242,5,low,F,4,1155,0
10997,10998,F,Ship,5,2,223,6,medium,M,2,1210,0


**1. KPIs de Gestión de Logística y Operaciones (Shipment & Warehouse)**

Permiten medir la eficiencia en la distribución de los productos electrónicos.

1.1. **Costo Promedio de Envío por Modo de Transporte (Mode of Shipment):** Ayuda a evaluar qué canales (Barco, Avión, Carretera) representan un mayor impacto financiero en relación con el peso del producto.

1.2. **Volumen de Ocupación por Bloque de Bodega (Warehouse Block):** Porcentaje o cantidad total de productos/peso administrados en cada bloque (A, B, C, D, E) para entender la carga operativa de la infraestructura.

1.3. **Peso Promedio del Producto por Modo de Envío:** Para analizar si se está optimizando la logística (ej. si los productos más pesados se están enviando por barco o si se está gastando de más enviándolos por avión).

In [31]:
print("=== 1.1. Costo Promedio del Producto y Peso por Modo de Envío ===")

logistica_envio = data.groupby('Mode_of_Shipment')[['Cost_of_the_Product', 'Weight_in_gms']].mean()
print(logistica_envio.round(2))

print("\n=== 1.2. Volumen de Ocupación y Peso Total por Bloque de Bodega ===")
bodega_ocupacion = data.groupby('Warehouse_block').agg(
    Total_Productos=('ID', 'count'),
    Peso_Total_Gramos=('Weight_in_gms', 'sum')
)
# Calculamos el porcentaje de ocupación en base al total de productos del dataset
bodega_ocupacion['Porcentaje_Ocupacion'] = (bodega_ocupacion['Total_Productos'] / len(data)) * 100
print(bodega_ocupacion.round(2))

print("\n=== 1.3. Peso Promedio del Producto por Modo de Envío ===")
peso_promedio = data.groupby('Mode_of_Shipment')['Weight_in_gms'].mean()
print(peso_promedio.round(2))

=== 1.1. Costo Promedio del Producto y Peso por Modo de Envío ===
                  Cost_of_the_Product  Weight_in_gms
Mode_of_Shipment                                    
Flight                         209.31        3629.38
Road                           210.48        3649.55
Ship                           210.34        3631.46

=== 1.2. Volumen de Ocupación y Peso Total por Bloque de Bodega ===
                 Total_Productos  Peso_Total_Gramos  Porcentaje_Ocupacion
Warehouse_block                                                          
A                           1833            6627118                 16.67
B                           1833            6664240                 16.67
C                           1833            6674560                 16.67
D                           1834            6655305                 16.67
F                           3666           13349327                 33.33

=== 1.3. Peso Promedio del Producto por Modo de Envío ===
Mode_of_Shipment
Flight

2. **KPIs de Gestión Comercial y Financiera (Sales & Revenue)**

Enfocados en el rendimiento del dinero, los descuentos y la estrategia de precios.

2.1. **Ingreso Total Bruto y Costo Promedio del Producto:** Suma y promedio de la variable Cost of the product para entender el ticket promedio de la tienda de electrónica.

2.2. **Porcentaje de Descuento Promedio Otorgado:** Evaluar el impacto de la estrategia de promociones (Discount offered) y si los descuentos altos realmente correlacionan con clientes de mayor prioridad o compras previas.

2.3. **Margen Teórico Relativo por Importancia del Producto:** Cruzar Product importance (Low, Medium, High) con el costo para ver si los productos catalogados como "alta importancia" son también los más costosos o los que reciben más descuentos.

In [32]:
print("=== 2.1. Ingreso Total Bruto y Costo Promedio del Producto ===")
ingreso_total = data['Cost_of_the_Product'].sum()
costo_promedio = data['Cost_of_the_Product'].mean()
print(f"Ingreso Total Bruto (Suma de Costos): ${ingreso_total:,}")
print(f"Ticket Costo Promedio: ${costo_promedio:.2f}")

print("\n=== 2.2. Porcentaje de Descuento Promedio Otorgado ===")
descuento_promedio = data['Discount_offered'].mean()
print(f"Descuento Promedio General: {descuento_promedio:.2f}%")


desc_por_compras = data.groupby('Prior_purchases')['Discount_offered'].mean()
print("\nDescuento promedio según compras previas del cliente:")
print(desc_por_compras.round(2))

print("\n=== 2.3. Margen Teórico y Descuento por Importancia del Producto ===")
importancia_analisis = data.groupby('Product_importance').agg(
    Costo_Promedio=('Cost_of_the_Product', 'mean'),
    Descuento_Promedio=('Discount_offered', 'mean'),
    Cantidad_Productos=('ID', 'count')
)
print(importancia_analisis.round(2))

=== 2.1. Ingreso Total Bruto y Costo Promedio del Producto ===
Ingreso Total Bruto (Suma de Costos): $2,311,955
Ticket Costo Promedio: $210.20

=== 2.2. Porcentaje de Descuento Promedio Otorgado ===
Descuento Promedio General: 13.37%

Descuento promedio según compras previas del cliente:
Prior_purchases
2     15.32
3     14.98
4     10.85
5      9.27
6     11.06
7     17.53
8     13.32
10    13.60
Name: Discount_offered, dtype: float64

=== 2.3. Margen Teórico y Descuento por Importancia del Producto ===
                    Costo_Promedio  Descuento_Promedio  Cantidad_Productos
Product_importance                                                        
high                        203.87               14.67                 948
low                         212.06               13.04                5297
medium                      209.38               13.48                4754


3. **KPIs de Experiencia y Fidelización del Cliente (Customer Experience - CX)**

Diseñados para evaluar la satisfacción y la carga de soporte post-venta.

3.1. **Índice de Satisfacción del Cliente (CSAT) Promedio:** El promedio de la variable Customer rating (escala 1 a 5). Se puede segmentar por género o por bloque de bodega para ver si hay anomalías.

3.2. **Tasa de Retención / Clientes Recurrentes:** Analizar la distribución de Prior purchases para determinar qué porcentaje de la base de datos son clientes nuevos (0-1 compras previas) vs. clientes fidelizados (2 o más compras).

3.3. **Carga de Soporte por Cliente (Customer Care Calls):** Promedio de llamadas de consulta realizadas. Permite analizar si a mayor costo del producto o menores calificaciones, aumentan las llamadas de reclamo.

In [33]:
print("=== 3.1. Índice de Satisfacción del Cliente (CSAT) Promedio ===")
csat_general = data['Customer_rating'].mean()
print(f"CSAT Promedio General (Escala 1-5): {csat_general:.2f}")


csat_por_genero = data.groupby('Gender')['Customer_rating'].mean()
print("\nCSAT Promedio por Género:")
print(csat_por_genero.round(2))

print("\n=== 3.2. Tasa de Retención / Clientes Recurrentes ===")

distribucion_compras = data['Prior_purchases'].value_counts().sort_index()
print("Distribución del número de compras previas por cliente:")
print(distribucion_compras)

print("\n=== 3.3. Carga de Soporte por Cliente (Customer Care Calls) ===")
llamadas_promedio = data['Customer_care_calls'].mean()
print(f"Promedio de llamadas de soporte por cliente: {llamadas_promedio:.2f}")


llamadas_vs_rating = data.groupby('Customer_rating')['Customer_care_calls'].mean()
print("\nPromedio de llamadas de soporte según la calificación del cliente (CSAT):")
print(llamadas_vs_rating.round(2))

=== 3.1. Índice de Satisfacción del Cliente (CSAT) Promedio ===
CSAT Promedio General (Escala 1-5): 2.99

CSAT Promedio por Género:
Gender
F    2.99
M    2.99
Name: Customer_rating, dtype: float64

=== 3.2. Tasa de Retención / Clientes Recurrentes ===
Distribución del número de compras previas por cliente:
Prior_purchases
2     2599
3     3955
4     2155
5     1287
6      561
7      136
8      128
10     178
Name: count, dtype: int64

=== 3.3. Carga de Soporte por Cliente (Customer Care Calls) ===
Promedio de llamadas de soporte por cliente: 4.05

Promedio de llamadas de soporte según la calificación del cliente (CSAT):
Customer_rating
1    4.04
2    4.05
3    4.06
4    4.04
5    4.09
Name: Customer_care_calls, dtype: float64
